# Phase 1 - Step 4: Data Relationships

This notebook analyzes cross-dataset relationships, join keys, and overlap statistics.

In [1]:
import pandas as pd
from pathlib import Path

processed_dir = Path("data/processed")
docs_dir = Path("docs")
docs_dir.mkdir(parents=True, exist_ok=True)

df_attr = pd.read_csv(processed_dir / "employee_attrition_processed.csv")
df_eng = pd.read_csv(processed_dir / "engagement_processed.csv")
df_occ = pd.read_csv(processed_dir / "occupation_master.csv")
df_ess = pd.read_csv(processed_dir / "essential_skills_processed.csv")
df_soft = pd.read_csv(processed_dir / "software_skills_processed.csv")

relationships = []

# 1. Attrition vs Engagement (Employee ID)
id_overlap = set(df_attr['EmployeeID']).intersection(set(df_eng['employee_id']))
relationships.append({
    "Dataset A": "employee_attrition_processed.csv",
    "Dataset B": "engagement_processed.csv",
    "Join Key": "EmployeeID <-> employee_id",
    "Relationship Type": "None (Disjoint Cohorts)",
    "Overlap Evidence": f"{len(id_overlap)} matching IDs (0% overlap, 1-500 vs 100021-999957)",
    "Purpose": "Attrition study cohort vs Organization-wide engagement monitoring cohort",
    "Confidence": "Definite (Disjoint ID spaces)"
})

# 2. Attrition vs Engagement (Job Roles / Departments)
role_overlap = set(df_attr['JobRole']).intersection(set(df_eng['job_role']))
dept_overlap = set(df_attr['Department']).intersection(set(df_eng['department']))
relationships.append({
    "Dataset A": "employee_attrition_processed.csv",
    "Dataset B": "engagement_processed.csv",
    "Join Key": "Department, JobRole <-> department, job_role",
    "Relationship Type": "Many-to-Many (Categorical Taxonomy)",
    "Overlap Evidence": f"5 common departments ({dept_overlap}), 5 exact common roles ({role_overlap})",
    "Purpose": "Cross-cohort departmental benchmarking and role modeling",
    "Confidence": "High"
})

# 3. Occupation Master vs Essential Skills
occ_codes = set(df_occ['O*NET-SOC Code'])
ess_codes = set(df_ess['O*NET-SOC Code'])
overlap_ess = occ_codes.intersection(ess_codes)
relationships.append({
    "Dataset A": "occupation_master.csv",
    "Dataset B": "essential_skills_processed.csv",
    "Join Key": "O*NET-SOC Code",
    "Relationship Type": "One-to-Many",
    "Overlap Evidence": f"{len(overlap_ess)} / {len(occ_codes)} occupations matched ({len(overlap_ess)/len(occ_codes)*100:.1f}%)",
    "Purpose": "Map baseline core soft/cognitive competencies to standardized occupations",
    "Confidence": "High"
})

# 4. Occupation Master vs Software Skills
soft_codes = set(df_soft['O*NET-SOC Code'])
overlap_soft = occ_codes.intersection(soft_codes)
relationships.append({
    "Dataset A": "occupation_master.csv",
    "Dataset B": "software_skills_processed.csv",
    "Join Key": "O*NET-SOC Code",
    "Relationship Type": "One-to-Many",
    "Overlap Evidence": f"{len(overlap_soft)} / {len(occ_codes)} occupations matched ({len(overlap_soft)/len(occ_codes)*100:.1f}%)",
    "Purpose": "Map technical software/tools requirements to standardized occupations",
    "Confidence": "High"
})

# 5. Essential Skills vs Software Skills
ess_soft_overlap = ess_codes.intersection(soft_codes)
relationships.append({
    "Dataset A": "essential_skills_processed.csv",
    "Dataset B": "software_skills_processed.csv",
    "Join Key": "O*NET-SOC Code",
    "Relationship Type": "Many-to-Many",
    "Overlap Evidence": f"{len(ess_soft_overlap)} shared O*NET codes",
    "Purpose": "Provide dual-dimension skill matrix (essential + technical tools) per occupation",
    "Confidence": "High"
})

rel_df = pd.DataFrame(relationships)
print(rel_df.to_string())


                          Dataset A                       Dataset B                                      Join Key                    Relationship Type                                                                                                                                                            Overlap Evidence                                                                           Purpose                     Confidence
0  employee_attrition_processed.csv        engagement_processed.csv                    EmployeeID <-> employee_id              None (Disjoint Cohorts)                                                                                                                         0 matching IDs (0% overlap, 1-500 vs 100021-999957)          Attrition study cohort vs Organization-wide engagement monitoring cohort  Definite (Disjoint ID spaces)
1  employee_attrition_processed.csv        engagement_processed.csv  Department, JobRole <-> department, job_role  Many-to-Many 

In [2]:
# Generate docs/data_relationships.md
md_content = """# Enterprise HR AI — Data Relationships and Schema Integration

This document defines the relationships, keys, overlap statistics, and integration architecture across all datasets in the platform.

## Relationship Matrix

| Dataset A | Dataset B | Join Key | Relationship Type | Overlap Evidence | Purpose | Confidence |
|---|---|---|---|---|---|---|
"""
for r in relationships:
    md_content += f"| `{r['Dataset A']}` | `{r['Dataset B']}` | `{r['Join Key']}` | {r['Relationship Type']} | {r['Overlap Evidence']} | {r['Purpose']} | {r['Confidence']} |\n"

md_content += """
## Key Architectural Decisions

1. **Employee ID Disjoint Cohorts**:
   - `employee_attrition_processed.csv` holds Employee IDs `1` to `500`.
   - `engagement_processed.csv` holds 5,000 distinct employees with 6-digit IDs (`100021` to `999957`).
   - These are separate employee datasets: one represents an attrition risk analysis study cohort with rich compensation, tenure, and exit signals; the other represents enterprise-wide performance and engagement tracking.
   - For unified employee intelligence, they are modeled harmoniously with consistent schema standards and role/department taxonomies.

2. **O*NET SOC Skill Taxonomy**:
   - `occupation_master.csv` serves as the canonical role dictionary (1,016 occupations).
   - `essential_skills_processed.csv` maps each occupation to 10 core competencies (Importance and Level ratings).
   - `software_skills_processed.csv` maps each occupation to specialized technical software, flagged with `Hot Technology` and `In Demand`.
"""

with open(docs_dir / "data_relationships.md", "w", encoding="utf-8") as f:
    f.write(md_content)

print(f"Generated {docs_dir / 'data_relationships.md'}")


Generated docs\data_relationships.md
